In [ ]:
#%pip install -q ultralytics opencv-python-headless scipy tqdm

In [ ]:
import os, json, math, csv
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
import matplotlib.pyplot as plt


BASE_UNZIPPED    = Path("/content/drive/MyDrive/pig_data_unzipped")
GLOBAL_MASK_PATH = BASE_UNZIPPED / "mask.png"

INDEX_ROOT = Path("/content/drive/MyDrive/pig_data_unzipped/pigs011219/PIGS011219")

N_FIRST_DIRS = 40

# ====== OUTPUT ======
SPLIT_NAME_1HZ = "behavior_1hz_video"
OUT_ROOT_1HZ   = Path(f"/content/drive/MyDrive/pig-frames_{SPLIT_NAME_1HZ}")
OUT_IMG_DIR    = OUT_ROOT_1HZ / "images_1hz"
OUT_COCO_JSON  = OUT_ROOT_1HZ / "annotate.coco.json"

OUT_ROOT_1HZ.mkdir(parents=True, exist_ok=True)
OUT_IMG_DIR.mkdir(parents=True, exist_ok=True)

TARGET_FPS   = 1.0
FPS_FALLBACK = 25.0

# ====== YOLO WEIGHTS ======
WEIGHTS      = "/content/weights.pt"
CONF_DET     = 0.25
IOU_DET      = 0.7
ALLOWED_CLASS = None

USE_MASK = True

print("INDEX_ROOT:", INDEX_ROOT)
print("OUT_ROOT_1HZ:", OUT_ROOT_1HZ)


# ================== HELPER ==================

def auto_collect_video_dirs(index_root: Path, n_first: int = 40):

    if not index_root.exists():
        pass
        return []

    subdirs = [p for p in index_root.iterdir() if p.is_dir()]

    def sort_key(p: Path):
        name = p.name
        return int(name) if name.isdigit() else name

    subdirs.sort(key=sort_key)

    video_dirs = []
    for d in subdirs:
        color_path = d / "color.mp4"
        if color_path.exists():
            video_dirs.append(d)
        if len(video_dirs) >= n_first:
            break

    pass
    for v in video_dirs:
        print("  -", v)
    return video_dirs


def load_mask(mask_path: Path):

    if not mask_path.exists():
        return None
    m = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if m is None:
        return None
    _, m = cv2.threshold(m, 127, 255, cv2.THRESH_BINARY)
    return m


def apply_mask_rgb(frame: np.ndarray, mask_gray: np.ndarray):

    if mask_gray is None:
        return frame
    H, W = frame.shape[:2]
    mh, mw = mask_gray.shape[:2]
    if (mh, mw) != (H, W):
        m = cv2.resize(mask_gray, (W, H), interpolation=cv2.INTER_NEAREST)
    else:
        m = mask_gray
    m3 = cv2.merge([m, m, m])
    return cv2.bitwise_and(frame, m3)


# ==== Appearance & geometry cho tracking ====

H_BINS, S_BINS, V_BINS = 16, 16, 4
APP_W, APP_H           = 96, 96

def extract_hist_hsv(frame, box):
    x1,y1,x2,y2 = map(int, box)
    Hh, Ww = frame.shape[:2]
    x1 = max(0, min(Ww-1, x1)); x2 = max(0, min(Ww-1, x2))
    y1 = max(0, min(Hh-1, y1)); y2 = max(0, min(Hh-1, y2))
    if x2 <= x1 or y2 <= y1:
        return np.ones((H_BINS*S_BINS*V_BINS,), np.float32)/(H_BINS*S_BINS*V_BINS)
    crop = frame[y1:y2, x1:x2]
    if crop.size == 0:
        return np.ones((H_BINS*S_BINS*V_BINS,), np.float32)/(H_BINS*S_BINS*V_BINS)
    crop = cv2.resize(crop, (APP_W, APP_H))
    hsv  = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist(
        [hsv],[0,1,2],None,
        [H_BINS,S_BINS,V_BINS],
        [0,180,0,256,0,256]
    ).astype(np.float32)
    hist /= (hist.sum() + 1e-6)
    return hist.flatten()

def bhattacharyya(h1, h2):
    return float(np.clip(1.0 - np.sum(np.sqrt(h1*h2)), 0.0, 1.0))

def iou_xyxy(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter = max(0, x2-x1)*max(0, y2-y1)
    ua = max(0, a[2]-a[0])*max(0, a[3]-a[1])
    ub = max(0, b[2]-b[0])*max(0, b[3]-b[1])
    return inter / (ua + ub - inter + 1e-6)


class Track:

    def __init__(self, tid, box, hist, frame_idx):
        self.tid = tid
        self.box = box
        self.hist_list = [hist]
        self.last_frame = frame_idx
        self.miss = 0
        self.age  = 1
        self.total_hits = 1
        self.fixed_id = None  # ID_1..ID_8

    def mean_hist(self):
        if not self.hist_list:
            return None
        return np.mean(np.stack(self.hist_list, 0), axis=0)


In [ ]:
TRACK_IOU_GATE = 0.1
TRACK_APP_GATE = 0.9
COST_THR       = 0.9
ALPHA_IOU      = 0.8
ALPHA_APP      = 1.0 - ALPHA_IOU

KEEPALIVE_MAX  = 30

MIN_HITS_FOR_ID = 5
MAX_IDS_PER_VIDEO = 8


# ====== TRACK & EXPORT ======
video_dirs = auto_collect_video_dirs(INDEX_ROOT, N_FIRST_DIRS)
mask_global = load_mask(GLOBAL_MASK_PATH) if USE_MASK else None
model = YOLO(str(WEIGHTS))

# COCO containers
images_list = []
annotations = []
categories  = [{"id": 1, "name": "Pig"}]
all_image_entries = {}   # img_path -> image_entry

next_image_id = 1
next_ann_id   = 1

for vdir in video_dirs:
    vdir = Path(vdir)
    color_path = vdir / "color.mp4"
    if not color_path.exists():
        pass
        continue

    cap = cv2.VideoCapture(str(color_path))
    if not cap.isOpened():
        pass
        continue

    native_fps = cap.get(cv2.CAP_PROP_FPS) or FPS_FALLBACK
    native_fps = float(native_fps) if native_fps > 0 else FPS_FALLBACK
    step = max(1, int(round(native_fps / TARGET_FPS)))

    nframes = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    dir_id  = vdir.name
    vid_stem = color_path.stem

    pass

    tracks = {}
    next_tid = 1
    used_ids = []

    frame_idx = -1
    sample_idx = 0

    pbar = tqdm(total=nframes, desc=f"Track {dir_id}", leave=False)

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame_idx += 1
        pbar.update(1)

        H, W = frame.shape[:2]
        if mask_global is not None:
            frame_in = apply_mask_rgb(frame, mask_global)
        else:
            frame_in = frame

        res = model(frame_in, conf=CONF_DET, iou=IOU_DET, verbose=False)[0]
        dets = []
        if res.boxes is not None and len(res.boxes) > 0:
            for b in res.boxes:
                cls = int(b.cls[0].cpu().item()) if b.cls is not None else 0
                if (ALLOWED_CLASS is not None) and (cls != ALLOWED_CLASS):
                    continue
                box = b.xyxy[0].cpu().numpy().tolist()
                score = float(b.conf[0].cpu().item())
                dets.append({"box": box, "score": score})

        for d in dets:
            d["hist"] = extract_hist_hsv(frame_in, d["box"])

        active_tids = list(tracks.keys())
        matches = {}
        unmatched_dets = set(range(len(dets)))
        unmatched_tracks = set(active_tids)

        if active_tids and dets:
            M = len(active_tids); N = len(dets)
            C = np.full((M,N), 1e3, np.float32)
            for i, tid in enumerate(active_tids):
                tr = tracks[tid]
                t_hist = tr.mean_hist()
                t_box  = tr.box
                for j, d in enumerate(dets):
                    box = d["box"]; hist = d["hist"]
                    iou = iou_xyxy(t_box, box)
                    app = bhattacharyya(t_hist, hist)
                    # gate
                    if (iou < TRACK_IOU_GATE) and (app > TRACK_APP_GATE):
                        continue
                    cost = ALPHA_IOU*(1.0 - iou) + ALPHA_APP*app
                    C[i,j] = cost
            row_ind, col_ind = linear_sum_assignment(C)
            for ri, cj in zip(row_ind, col_ind):
                if C[ri, cj] > COST_THR:
                    continue
                tid = active_tids[ri]
                matches[tid] = cj
                unmatched_dets.discard(cj)
                unmatched_tracks.discard(tid)

        # 2) UPDATE TRACKS MATCHED
        for tid, j in matches.items():
            d = dets[j]
            tr = tracks[tid]
            tr.box = d["box"]
            tr.hist_list.append(d["hist"])
            tr.last_frame = frame_idx
            tr.miss = 0
            tr.total_hits += 1

        for j in unmatched_dets:
            d = dets[j]
            box = d["box"]; hist = d["hist"]
            tr = Track(next_tid, box, hist, frame_idx)
            tracks[next_tid] = tr
            next_tid += 1

        to_delete = []
        for tid in unmatched_tracks:
            tr = tracks[tid]
            tr.miss += 1
            tr.last_frame = frame_idx
            if tr.miss > KEEPALIVE_MAX:
                to_delete.append(tid)
        for tid in to_delete:
            del tracks[tid]

        for tr in tracks.values():
            if tr.fixed_id is None and tr.total_hits >= MIN_HITS_FOR_ID:
                if len(used_ids) < MAX_IDS_PER_VIDEO:
                    new_id = len(used_ids) + 1  # 1..8
                    tr.fixed_id = new_id
                    used_ids.append(new_id)
                else:
                    pass

        # ===== SAMPLING 1Hz =====
        if frame_idx % step != 0:
            continue

        t_sec = frame_idx / native_fps
        ms    = int(t_sec * 1000)
        out_name = f"{dir_id}_{vid_stem}_t{ms:06d}_idx{sample_idx:05d}.jpg"
        out_path = OUT_IMG_DIR / out_name

        ok_w = cv2.imwrite(str(out_path), frame_in)
        if not ok_w:
            pass
            sample_idx += 1
            continue

        if str(out_path) not in all_image_entries:
            Hh, Ww = frame_in.shape[:2]
            img_id = next_image_id
            next_image_id += 1
            img_entry = {
                "id": img_id,
                "file_name": out_name,
                "width": Ww,
                "height": Hh,
                "video": str(color_path),
                "frame_idx": int(frame_idx),
                "time_s": round(float(t_sec), 3)
            }
            all_image_entries[str(out_path)] = img_entry
            images_list.append(img_entry)
        else:
            img_id = all_image_entries[str(out_path)]["id"]

        for tr in tracks.values():
            if tr.fixed_id is None:
                continue
            x1,y1,x2,y2 = tr.box
            w = max(0.0, x2-x1); h = max(0.0, y2-y1)
            attr = {
                "ID": f"ID_{tr.fixed_id}",
                "Behavior": "lying",
                "Hidden": "No" if tr.miss == 0 else "Yes"
            }
            ann = {
                "id": next_ann_id,
                "image_id": img_id,
                "category_id": 1,
                "bbox": [
                    round(float(x1),2),
                    round(float(y1),2),
                    round(float(w),2),
                    round(float(h),2)
                ],
                "iscrowd": 0,
                "attributes": attr
            }
            annotations.append(ann)
            next_ann_id += 1

        sample_idx += 1

    pbar.close()
    cap.release()

# ====== GHI COCO ======
coco = {
    "images": images_list,
    "annotations": annotations,
    "categories": categories
}
with open(OUT_COCO_JSON, "w", encoding="utf-8") as f:
    json.dump(coco, f, ensure_ascii=False)

pass
pass
print("  - COCO:", OUT_COCO_JSON)
pass
pass


In [ ]:
# Load COCO
with open(OUT_COCO_JSON, "r", encoding="utf-8") as f:
    coco = json.load(f)

imgid_to_img = {im["id"]: im for im in coco["images"]}
imgid_to_anns = {}
for ann in coco["annotations"]:
    imgid_to_anns.setdefault(ann["image_id"], []).append(ann)

video_groups = {}
for im in coco["images"]:
    vid = im.get("video", "unknown")
    video_groups.setdefault(vid, []).append(im)

target_vid = None
for vid, ims in video_groups.items():
    if len(ims) >= 15:
        target_vid = vid
        break

if target_vid is None:
    pass
else:
    ims = sorted(video_groups[target_vid], key=lambda x: x["frame_idx"])
    ims_15 = ims[:15]
    pass

    plt.figure(figsize=(5*3, 3*3))
    for i, im in enumerate(ims_15):
        img_id = im["id"]
        file_name = im["file_name"]
        img_path = OUT_IMG_DIR / file_name
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img_rgb = img.copy()

        anns = imgid_to_anns.get(img_id, [])

        for ann in anns:
            x,y,w,h = ann["bbox"]
            x1,y1,x2,y2 = int(x), int(y), int(x+w), int(y+h)
            attr = ann.get("attributes", {})
            id_str = attr.get("ID", "")
            hidden = attr.get("Hidden", "No")
            color = (37*hash(id_str)%255, 97*hash(id_str)%255, 173*hash(id_str)%255)
            cv2.rectangle(img_rgb, (x1,y1), (x2,y2), color, 2)
            label = id_str if hidden=="No" else f"{id_str} (H)"
            cv2.putText(img_rgb, label, (x1, max(0,y1-5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)

        img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2RGB)
        ax = plt.subplot(3, 5, i+1)
        ax.imshow(img_rgb); ax.set_axis_off()
        ax.set_title(file_name, fontsize=8)

    plt.tight_layout()
    plt.show()
